# Task 1 - Dataset Exploration and Understanding
- Key findings
  - Overrepresentation of eye features 
  - No duplicate or missing rows
  - 10,000 images appear to be missing

In [3]:
#Importing relevant libraries
import os
import pandas as pd
from PIL import Image

#==============================================
#================Initialization================

#Reading csv file / Defining paths
train_dir = "train/train"
test_dir = "test"
labels = "train_labels.csv"
df = pd.read_csv(labels)


print(df.head(), "\n") #Displaying first 5 rows
print("Rows:", df.shape[0], " Columns:", df.shape[1], "\n")  #Counting rows and columns

#==============================================
#==========Class Distribution Analysis=========

#Adding new column to dataframe with filepaths to images
df['filepath'] = df['image_id'].apply(
    lambda x: os.path.join(train_dir, f"{x}.jpg")
)

# Compute counts and percentages
class_counts = df['class'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']
class_counts['Percentage (%)'] = (class_counts['Count'] / len(df) * 100).round(2)

print("=== Class Distribution Summary ===")
print(class_counts.to_string(index=False))

# Calculate Imbalance Ratio (Max / Min)
imbalance_ratio = class_counts['Count'].max() / class_counts['Count'].min()
print(f"\nClass Imbalance Ratio (CatEye / DogMouth): {imbalance_ratio:.2f}")

#======================================================
#==========Image Quality & Resolution Analysis=========

import os
import pandas as pd
from PIL import Image

corrupt_files = []
image_metadata = []

print("=== DIRECTORY DIAGNOSTICS ===")
print("Current Working Directory:", os.getcwd())

# 1. Verify train folder existence
if os.path.exists(train_dir):
    files_in_train = os.listdir(train_dir)
    print(f"Found '{train_dir}' directory containing {len(files_in_train)} items.")
    if files_in_train:
        print("Sample file on disk:", files_in_train[0])
else:
    print(f"[WARNING] Directory '{train_dir}' was NOT found in the current folder!")

# 2. Iterate through dataframe and inspect images safely
for idx, row in df.iterrows():
    path = row['filepath']
    
    # Check if path exists; try alternative extensions if missing
    if not os.path.exists(path):
        base_path, _ = os.path.splitext(path)
        found_alt = False
        for ext in ['.png', '.jpeg', '.JPG', '.PNG']:
            alt_path = base_path + ext
            if os.path.exists(alt_path):
                path = alt_path
                found_alt = True
                break
        
        if not found_alt:
            corrupt_files.append((path, "File missing from disk"))
            continue

    try:
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
            image_metadata.append({
                'image_id': row['image_id'],
                'class': row['class'],
                'width': width,
                'height': height,
                'aspect_ratio': round(width / height, 2),
                'mode': mode
            })
    except Exception as e:
        corrupt_files.append((path, str(e)))

# 3. Create metadata DataFrame
df_meta = pd.DataFrame(image_metadata)

print("\n=== IMAGE QUALITY & RESOLUTION SUMMARY ===")
print(f"Total valid images checked: {len(df_meta)}")
print(f"Corrupt / missing files detected: {len(corrupt_files)}")

# 4. Display metadata summary statistics safely
if not df_meta.empty:
    print("\nResolution Statistics:")
    print(df_meta[['width', 'height', 'aspect_ratio']].describe())
    print("\nColor Channel Modes present:", df_meta['mode'].unique())
else:
    print("\n[ERROR] No valid images were loaded into df_meta!")
    if corrupt_files:
        print("Sample missing file path logged:", corrupt_files[0][0])
        print("Tip: Ensure your 'train_dir' folder is in the same directory as your notebook.")

   image_id     class
0         1    DogEye
1         2  DogMouth
2         3   DogFace
3         4   DogNose
4         5   DogNose 

Rows: 63660  Columns: 2 

=== Class Distribution Summary ===
   Class  Count  Percentage (%)
  CatEye  13884           21.81
  DogEye  11580           18.19
 CatNose   6942           10.90
 CatFace   6942           10.90
CatMouth   6942           10.90
DogMouth   5790            9.10
 DogFace   5790            9.10
 DogNose   5790            9.10

Class Imbalance Ratio (CatEye / DogMouth): 2.40
=== DIRECTORY DIAGNOSTICS ===
Current Working Directory: /Users/arnoldcave/Desktop/Kaggle Competition/enge707-facial-feature-classification
Found 'train/train' directory containing 63660 items.
Sample file on disk: 14147.jpg

=== IMAGE QUALITY & RESOLUTION SUMMARY ===
Total valid images checked: 53661
Corrupt / missing files detected: 9999

Resolution Statistics:
              width        height  aspect_ratio
count  53661.000000  53661.000000  53661.000000
mean  

# Task 2: Data Preprocessing and Pipeline Preparation

In [13]:
#======================================================
#==========Label Encoding and Stratified Split=========

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Ensure df_clean only contains existing files
df['filepath'] = df['image_id'].apply(lambda x: os.path.join(train_dir, f"{x}.jpg"))
df_clean = df[df['filepath'].apply(os.path.exists)].reset_index(drop=True)

# 2. Encode text class labels into numerical integers (0 to 7)
class_names = sorted(df_clean['class'].unique())
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}

df_clean['label'] = df_clean['class'].map(label_to_idx)

print("=== Class Mapping ===")
for name, idx in label_to_idx.items():
    print(f"  Class '{name}' -> Label {idx}")

# 3. Perform Stratified Train/Val Split (80% Train, 20% Validation)
train_df, val_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=42,
    stratify=df_clean['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\n=== Dataset Split Summary ===")
print(f"Total Valid Images : {len(df_clean)}")
print(f"Training Set (80%) : {len(train_df)}")
print(f"Validation Set (20%): {len(val_df)}")

#===========================================================
#==========Image Augmentation and PyToch Transforms=========

import torch
from torchvision import transforms

# Standard ImageNet Normalization Constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training Transforms (With Data Augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation & Testing Transforms (No Augmentation, Deterministic Resizing)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

#===========================================================
#===============Building Custom PyTorch Class===============

from torch.utils.data import Dataset
from PIL import Image

class FacialFeatureDataset(Dataset):
    """Custom PyTorch Dataset for ENGE707 Facial Feature Images."""
    
    def __init__(self, dataframe, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['filepath']
        
        # Safely open image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, row['image_id']
        else:
            label = row['label']
            return image, torch.tensor(label, dtype=torch.long)

#===========================================================
#================Creating PyTorch DataLoader================

from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 0  # Set to 2 for macOS CPU/MPS execution

# 1. Instantiate Dataset objects
train_dataset = FacialFeatureDataset(train_df, transform=train_transforms)
val_dataset   = FacialFeatureDataset(val_df, transform=val_transforms)

# 2. Instantiate DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# 3. Verify Batch Shape
sample_images, sample_labels = next(iter(train_loader))
print("=== DataLoader Verification ===")
print(f"Batch Images Shape : {sample_images.shape}")  # Should be [32, 3, 224, 224]
print(f"Batch Labels Shape : {sample_labels.shape}")  # Should be [32]

=== Class Mapping ===
  Class 'CatEye' -> Label 0
  Class 'CatFace' -> Label 1
  Class 'CatMouth' -> Label 2
  Class 'CatNose' -> Label 3
  Class 'DogEye' -> Label 4
  Class 'DogFace' -> Label 5
  Class 'DogMouth' -> Label 6
  Class 'DogNose' -> Label 7

=== Dataset Split Summary ===
Total Valid Images : 53661
Training Set (80%) : 42928
Validation Set (20%): 10733
=== DataLoader Verification ===
Batch Images Shape : torch.Size([32, 3, 224, 224])
Batch Labels Shape : torch.Size([32])
